In [26]:
import pandas as pd

base = r"F:\M.Sc in Economics\Thesis paper"

# ------------------------------------------------
# 1. LOAD PANEL DATA WITH CONTROLS
# ------------------------------------------------
panel = pd.read_csv(base + r"\panel_dataset_controls.csv")

# clean panel types
panel["kldb3"] = pd.to_numeric(panel["kldb3"], errors="coerce").astype("Int64")
panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype("Int64")
panel["bestand_absolut"] = pd.to_numeric(panel["bestand_absolut"], errors="coerce")
panel["vakanz_tage"] = pd.to_numeric(panel["vakanz_tage"], errors="coerce")
panel["relation"] = pd.to_numeric(panel["relation"], errors="coerce")
panel["log_bestand"] = pd.to_numeric(panel["log_bestand"], errors="coerce")
panel["log_vakanz"] = pd.to_numeric(panel["log_vakanz"], errors="coerce")
panel["log_relation"] = pd.to_numeric(panel["log_relation"], errors="coerce")

panel = panel.dropna(
    subset=[
        "kldb3",
        "year",
        "bestand_absolut",
        "vakanz_tage",
        "relation",
        "log_bestand",
        "log_vakanz",
        "log_relation"
    ]
).copy()

print("Panel columns:")
print(panel.columns.tolist())
print(panel.head())


# ------------------------------------------------
# 2. LOAD AI EXPOSURE DATA
# ------------------------------------------------
ai = pd.read_excel(base + r"\AIOE_DataAppendix.xlsx", sheet_name="Appendix A")

ai = ai.rename(columns={
    "SOC Code": "soc",
    "Occupation Title": "occupation_title",
    "AIOE": "ai_exposure"
})

ai = ai[["soc", "occupation_title", "ai_exposure"]].copy()

# clean SOC codes
ai["soc"] = ai["soc"].astype(str).str.replace("-", "", regex=False)
ai["soc_major"] = ai["soc"].str[:2]
ai["ai_exposure"] = pd.to_numeric(ai["ai_exposure"], errors="coerce")

ai = ai.dropna(subset=["soc_major", "ai_exposure"]).copy()

# average AI exposure by SOC major group
ai_major = ai.groupby("soc_major", as_index=False)["ai_exposure"].mean()

print("\nAI preview:")
print(ai.head())

print("\nAI exposure by SOC major:")
print(ai_major.head())


# ------------------------------------------------
# 3. KLDB -> SOC CROSSWALK
# ------------------------------------------------
crosswalk = {
    111:45,112:45,113:45,115:45,117:45,121:45,122:39,
    211:47,212:51,221:51,222:51,223:51,231:51,232:27,234:51,
    241:51,242:51,243:51,244:51,245:51,
    251:17,252:17,261:17,262:17,263:17,271:19,272:17,273:13,
    281:51,282:51,283:51,291:51,292:51,293:35,
    311:17,312:17,321:47,322:47,331:47,332:47,333:47,
    341:49,342:47,343:51,
    411:15,412:19,413:19,414:19,421:19,422:19,423:19,
    431:15,432:15,433:15,434:15,
    511:53,512:53,513:53,514:53,515:53,516:43,521:53,522:53,524:53,525:53,
    531:33,532:33,533:33,541:37,
    611:41,612:41,613:41,621:41,622:41,623:41,624:41,
    631:39,632:35,633:35,634:39,
    713:13,714:43,715:13,721:13,722:13,723:13,731:23,732:43,733:43,
    811:31,812:29,813:29,814:29,815:29,816:29,817:29,818:29,
    821:31,822:29,823:39,825:29,
    831:21,832:39,841:25,842:25,843:25,844:25,845:25,
    912:19,913:19,921:13,922:27,924:27,932:27,935:27,941:27,942:27
}

panel["soc_major"] = panel["kldb3"].map(crosswalk)
panel = panel.dropna(subset=["soc_major"]).copy()
panel["soc_major"] = panel["soc_major"].astype(int).astype(str)

print("\nPanel with SOC mapping:")
print(panel.head())


# ------------------------------------------------
# 4. MERGE PANEL + AI EXPOSURE
# ------------------------------------------------
final = panel.merge(ai_major, on="soc_major", how="left")

print("\nMerged preview:")
print(final.head())
print("\nMissing AI exposure share:", final["ai_exposure"].isna().mean())


# ------------------------------------------------
# 5. COLLAPSE DUPLICATES AND KEEP ALL COLUMNS
# ------------------------------------------------
final = (
    final.groupby(["kldb3", "year", "soc_major"], as_index=False)
    .agg({
        "bestand_absolut": "mean",
        "vakanz_tage": "mean",
        "relation": "mean",
        "log_bestand": "mean",
        "log_vakanz": "mean",
        "log_relation": "mean",
        "ai_exposure": "mean"
    })
)

print("\nFinal columns:")
print(final.columns.tolist())

print("\nFinal preview:")
print(final.head(15))


# ------------------------------------------------
# 6. SAVE
# ------------------------------------------------
out = base + r"\final_dataset_ai_exposure_controls.csv"

final.to_csv(out, index=False)

print("\nSaved:", out)

Panel columns:
['kldb3', 'year', 'bestand_absolut', 'vakanz_tage', 'relation', 'log_bestand', 'log_vakanz', 'log_relation']
   kldb3  year  bestand_absolut  vakanz_tage  relation  log_bestand  \
0    111  2015            478.0         68.0     538.0     6.169611   
1    111  2016            486.0         74.0     562.0     6.186209   
2    111  2017            598.0         77.0     432.0     6.393591   
3    111  2018            797.0         91.0     288.0     6.680855   
4    111  2019            793.0        106.0     269.0     6.675823   

   log_vakanz  log_relation  
0    4.219508      6.287859  
1    4.304065      6.331502  
2    4.343805      6.068426  
3    4.510860      5.662960  
4    4.663439      5.594711  

AI preview:
      soc                     occupation_title  ai_exposure soc_major
0  111011                     Chief Executives     1.334246        11
1  111021      General and Operations Managers     0.574877        11
2  112011  Advertising and Promotions Managers

In [3]:
import pandas as pd
from linearmodels.panel import PanelOLS

base = r"F:\M.Sc in Economics\Thesis paper"

df = pd.read_csv(base+r"\final_dataset_ai_exposure_controls.csv")

# -------------------------
# Panel index
# -------------------------
df = df.set_index(["kldb3","year"])

# Post period dummy
df["post"] = (df.index.get_level_values("year") >= 2023).astype(int)

# Continuous DiD interaction
df["ai_post"] = df["ai_exposure"] * df["post"]

print(df[["ai_exposure","post","ai_post"]].head(15))

            ai_exposure  post   ai_post
kldb3 year                             
111   2015    -1.056699     0 -0.000000
      2016    -1.056699     0 -0.000000
      2017    -1.056699     0 -0.000000
      2018    -1.056699     0 -0.000000
      2019    -1.056699     0 -0.000000
      2021    -1.056699     0 -0.000000
      2022    -1.056699     0 -0.000000
      2023    -1.056699     1 -1.056699
      2024    -1.056699     1 -1.056699
      2025    -1.056699     1 -1.056699
112   2015    -1.056699     0 -0.000000
      2016    -1.056699     0 -0.000000
      2017    -1.056699     0 -0.000000
      2018    -1.056699     0 -0.000000
      2019    -1.056699     0 -0.000000


In [4]:
model1 = PanelOLS.from_formula(
    "log_relation ~ ai_exposure + TimeEffects",
    data=df
)

res1 = model1.fit(cov_type="clustered", cluster_entity=True)

print(res1)

                          PanelOLS Estimation Summary                           
Dep. Variable:           log_relation   R-squared:                        0.0448
Estimator:                   PanelOLS   R-squared (Between):              0.0018
No. Observations:                1253   R-squared (Within):               0.0000
Date:                Thu, Apr 30 2026   R-squared (Overall):              0.0023
Time:                        13:40:31   Log-likelihood                   -1855.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      58.208
Entities:                         117   P-value                           0.0000
Avg Obs:                       10.709   Distribution:                  F(1,1241)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             10.457
                            

In [5]:
model2 = PanelOLS.from_formula(
    "log_relation ~ ai_post + TimeEffects",
    data=df
)

res2 = model2.fit(cov_type="clustered", cluster_entity=True)

print(res2)

                          PanelOLS Estimation Summary                           
Dep. Variable:           log_relation   R-squared:                        0.0278
Estimator:                   PanelOLS   R-squared (Between):              0.0003
No. Observations:                1253   R-squared (Within):               0.0003
Date:                Thu, Apr 30 2026   R-squared (Overall):              0.0009
Time:                        13:40:32   Log-likelihood                   -1866.1
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      35.512
Entities:                         117   P-value                           0.0000
Avg Obs:                       10.709   Distribution:                  F(1,1241)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             9.0831
                            

In [6]:
model3 = PanelOLS.from_formula(
    """
    log_relation ~
    ai_exposure
    + ai_post
    + log_bestand
    + TimeEffects
    """,
    data=df
)

res3 = model3.fit(
    cov_type="clustered",
    cluster_entity=True
)

print(res3)

                          PanelOLS Estimation Summary                           
Dep. Variable:           log_relation   R-squared:                        0.1769
Estimator:                   PanelOLS   R-squared (Between):              0.4904
No. Observations:                1253   R-squared (Within):               0.0128
Date:                Thu, Apr 30 2026   R-squared (Overall):              0.4695
Time:                        13:40:32   Log-likelihood                   -1761.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      88.748
Entities:                         117   P-value                           0.0000
Avg Obs:                       10.709   Distribution:                  F(3,1239)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             20.737
                            

In [7]:
print(df[["log_bestand"]].head())

            log_bestand
kldb3 year             
111   2015     6.169611
      2016     6.186209
      2017     6.393591
      2018     6.680855
      2019     6.675823


In [8]:
from linearmodels.panel import PanelOLS

model = PanelOLS.from_formula(
"""
log_relation ~
ai_exposure
+ ai_post
+ log_bestand
+ TimeEffects
""",
data=df
)

res=model.fit(
cov_type="clustered",
cluster_entity=True
)

print(res)

                          PanelOLS Estimation Summary                           
Dep. Variable:           log_relation   R-squared:                        0.1769
Estimator:                   PanelOLS   R-squared (Between):              0.4904
No. Observations:                1253   R-squared (Within):               0.0128
Date:                Thu, Apr 30 2026   R-squared (Overall):              0.4695
Time:                        13:40:32   Log-likelihood                   -1761.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      88.748
Entities:                         117   P-value                           0.0000
Avg Obs:                       10.709   Distribution:                  F(3,1239)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             20.737
                            

In [9]:
print(df["log_bestand"].describe())

count    1253.000000
mean        7.393908
std         1.418192
min         2.302585
25%         6.261492
50%         7.361058
75%         8.546946
max        10.534646
Name: log_bestand, dtype: float64


In [10]:
print(df["log_bestand"].var())

2.0112681376816584


In [11]:

print(df[["log_bestand","ai_exposure"]].corr())

             log_bestand  ai_exposure
log_bestand     1.000000     0.080885
ai_exposure     0.080885     1.000000


In [12]:
import pandas as pd
from linearmodels.panel import PanelOLS

base = r"F:\M.Sc in Economics\Thesis paper"

# load the correct updated file
df = pd.read_csv(base + r"\final_dataset_ai_exposure_controls.csv")

# clean column names
df.columns = df.columns.str.strip()

print(df.columns.tolist())

# make sure numeric
for col in ["kldb3", "year", "log_relation", "log_bestand", "ai_exposure"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# create post and ai_post
df["post"] = (df["year"] >= 2023).astype(int)
df["ai_post"] = df["ai_exposure"] * df["post"]

# drop missing
df = df.dropna(subset=[
    "kldb3",
    "year",
    "log_relation",
    "ai_exposure",
    "ai_post",
    "log_bestand"
])

# set panel index
df = df.set_index(["kldb3", "year"])

# run controlled model
model = PanelOLS.from_formula(
    "log_relation ~ ai_exposure + ai_post + log_bestand + TimeEffects",
    data=df
)

results = model.fit(cov_type="clustered", cluster_entity=True)

print(results.summary)

['kldb3', 'year', 'soc_major', 'bestand_absolut', 'vakanz_tage', 'relation', 'log_bestand', 'log_vakanz', 'log_relation', 'ai_exposure']
                          PanelOLS Estimation Summary                           
Dep. Variable:           log_relation   R-squared:                        0.1769
Estimator:                   PanelOLS   R-squared (Between):              0.4904
No. Observations:                1253   R-squared (Within):               0.0128
Date:                Thu, Apr 30 2026   R-squared (Overall):              0.4695
Time:                        13:40:33   Log-likelihood                   -1761.8
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      88.748
Entities:                         117   P-value                           0.0000
Avg Obs:                       10.709   Distribution:                  F(3,1239)
Min Obs:                       1.0000                

In [13]:
"kldb3"


'kldb3'

In [14]:
"ai_exposure"

'ai_exposure'

In [15]:
panel.columns.tolist()


['kldb3',
 'year',
 'bestand_absolut',
 'vakanz_tage',
 'relation',
 'log_bestand',
 'log_vakanz',
 'log_relation',
 'soc_major']

In [17]:
panel.columns = panel.columns.str.strip()


In [18]:
[c for c in panel.columns if "exposure" in c.lower() or "ai" in c.lower()]


[]

In [20]:
print(panel.columns.tolist())
print("ai_exposure" in panel.columns)
print(panel[["year", "post"]].head())


['kldb3', 'year', 'bestand_absolut', 'vakanz_tage', 'relation', 'log_bestand', 'log_vakanz', 'log_relation', 'soc_major', 'post']
False
   year  post
0  2015     0
1  2016     0
2  2017     0
3  2018     0
4  2019     0


In [21]:
panel.columns = (
    panel.columns
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(panel.columns.tolist())
[c for c in panel.columns if "ai" in c or "exposure" in c]


['kldb3', 'year', 'bestand_absolut', 'vakanz_tage', 'relation', 'log_bestand', 'log_vakanz', 'log_relation', 'soc_major', 'post']


[]

In [22]:
panel.filter(regex="ai|exposure", axis=1).head()


""
0
1
2
3
4


In [35]:
import pyfixest
print(pyfixest.__version__)
print(dir(pyfixest))

0.50.1
['SaturatedEventStudy', 'bonferroni', 'coefplot', 'did', 'did2s', 'dtable', 'errors', 'estimation', 'etable', 'event_study', 'feglm', 'feols', 'fepois', 'get_data', 'get_motherhood_event_study_data', 'get_ssc', 'get_twin_data', 'get_worker_panel', 'iplot', 'lpdid', 'panelview', 'qplot', 'quantreg', 'report', 'rwolf', 'ssc', 'summary', 'utils', 'wyoung']


In [36]:
import pyfixest as pf
print(dir(pf))

['SaturatedEventStudy', 'bonferroni', 'coefplot', 'did', 'did2s', 'dtable', 'errors', 'estimation', 'etable', 'event_study', 'feglm', 'feols', 'fepois', 'get_data', 'get_motherhood_event_study_data', 'get_ssc', 'get_twin_data', 'get_worker_panel', 'iplot', 'lpdid', 'panelview', 'qplot', 'quantreg', 'report', 'rwolf', 'ssc', 'summary', 'utils', 'wyoung']
